# Prepare Qwen3 30B A3B for TensorRT-LLM LLM API

Downloads `Qwen/Qwen3-30B-A3B` into the Triton Python backend model version folder. The deployed `model.py` loads it with TensorRT-LLM's LLM API.


In [ ]:
import os
import tempfile
from pathlib import Path

EXAMPLE_DIR_NAME = "qwen3-30b-a3b-llmapi-s3"
PROJECT = Path.cwd().resolve()
if PROJECT.name != EXAMPLE_DIR_NAME:
    raise RuntimeError(f"Run this notebook from {EXAMPLE_DIR_NAME}, got {PROJECT}")
if PROJECT.parent.name == EXAMPLE_DIR_NAME:
    raise RuntimeError(f"Nested example folder is wrong: {PROJECT}. Use one {EXAMPLE_DIR_NAME} folder only.")

MODEL_NAME = "qwen3_30b_a3b_llmapi"
HF_MODEL_ID = "Qwen/Qwen3-30B-A3B"
VERSION_DIR = PROJECT / MODEL_NAME / "1"
MODEL_DIR = VERSION_DIR / "model"
RUNTIME_DEPS_DIR = VERSION_DIR / "python_deps"
NOTEBOOK_WORK = Path(tempfile.mkdtemp(prefix="qwen3_30b_llmapi_notebook_"))
LOCAL_HOME = NOTEBOOK_WORK / "home"
LOCAL_CACHE = NOTEBOOK_WORK / "cache"
HF_CACHE = LOCAL_CACHE / "huggingface"
PIP_CACHE = LOCAL_CACHE / "pip"
for path in (LOCAL_HOME, LOCAL_CACHE, HF_CACHE, PIP_CACHE):
    path.mkdir(parents=True, exist_ok=True)

os.environ["USER"] = "workspace"
os.environ["LOGNAME"] = "workspace"
os.environ["HOME"] = str(LOCAL_HOME)
os.environ["XDG_CACHE_HOME"] = str(LOCAL_CACHE)
os.environ["HF_HOME"] = str(HF_CACHE)
os.environ["PIP_CACHE_DIR"] = str(PIP_CACHE)
os.environ["HF_HUB_DISABLE_XET"] = "1"
os.environ["PYTHONNOUSERSITE"] = "1"

print("PROJECT:", PROJECT)
print("VERSION_DIR:", VERSION_DIR)
print("MODEL_DIR:", MODEL_DIR)


In [ ]:
import subprocess
import sys

DEPS_DIR = NOTEBOOK_WORK / "deps"
DEPS_DIR.mkdir(exist_ok=True)

subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--upgrade",
    "--target",
    str(DEPS_DIR),
    "huggingface_hub>=0.23",
    "requests>=2.32",
    "idna>=3",
    "urllib3>=2",
    "certifi",
    "charset_normalizer",
    "tqdm",
    "filelock",
    "fsspec",
    "packaging",
    "pyyaml",
    "typing_extensions",
])

deps_path = str(DEPS_DIR)
if deps_path not in sys.path:
    sys.path.insert(0, deps_path)

print("Notebook deps:", DEPS_DIR)


In [ ]:
RUNTIME_DEPS_DIR.mkdir(exist_ok=True)
subprocess.check_call([
    sys.executable,
    "-m",
    "pip",
    "install",
    "--upgrade",
    "--target",
    str(RUNTIME_DEPS_DIR),
    "openai==2.44.0",
])

print("Runtime deps:", RUNTIME_DEPS_DIR)


In [ ]:
from huggingface_hub import snapshot_download

MODEL_DIR.mkdir(parents=True, exist_ok=True)
snapshot_download(
    repo_id=HF_MODEL_ID,
    local_dir=MODEL_DIR,
    local_dir_use_symlinks=False,
)

print("Downloaded files:")
for path in sorted(MODEL_DIR.iterdir()):
    print(path.name)


In [ ]:
required = ["config.json", "tokenizer_config.json"]
missing = [name for name in required if not (MODEL_DIR / name).exists()]
weights = list(MODEL_DIR.glob("*.safetensors")) + list(MODEL_DIR.glob("*.bin"))
if missing:
    raise RuntimeError(f"Missing required files: {missing}")
if not weights:
    raise RuntimeError(f"No model weights found in {MODEL_DIR}")
print("OK:", MODEL_DIR)
print("Weight files:", len(weights))
print("First weights:", [path.name for path in weights[:5]])


In [ ]:
import shutil

for path in (NOTEBOOK_WORK,):
    shutil.rmtree(path, ignore_errors=True)
    print("Cleaned:", path)
